### Initialization

In [ ]:
# Installing all required dependencies
%pip install gdown -q
%pip install scikit-learn
%pip install datasets
%pip install "transformers==4.47.1"
%pip install "peft==0.14.0"
%pip install accelerate
%pip install torchvision
%pip install torchaudio
%pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [21]:
import sys

# add path to analysis directory
# stays empty if using git directory structure
git_dir = ''

sys.path.insert(0, git_dir)

import importlib
import analysis.data
import analysis.prompts
import analysis.transformer_load_train
import analysis.llm_load_train
import analysis.evaluate
import analysis.predict
import analysis.build

# Reloads all directories in case any changes are made
importlib.reload(analysis.prompts)
importlib.reload(analysis.data)
importlib.reload(analysis.transformer_load_train)
importlib.reload(analysis.llm_load_train)
importlib.reload(analysis.evaluate)
importlib.reload(analysis.predict)
importlib.reload(analysis.build)

<module 'analysis.build' from 'c:\\Users\\setht\\OneDrive\\Desktop\\nation_strategy_text_data_paper\\national-strategy-text-data\\analysis\\build.py'>

In [ ]:
# login to huggengface, token required
import huggingface_hub
huggingface_hub.login(token=open("hf_token.txt").read())

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
!mkdir data
!mkdir output
!mkdir models

In [ ]:
data_path = "data"
results_path = "output"
model_path = "models"

In [ ]:
# running transformer models

transformer_model_sources = ["kornosk/polibertweet-political-twitter-roberta-mlm", #note: limited by 128 context window
                             "facebook/bart-large-mnli",
                             "mlburnham/Political_DEBATE_DeBERTa_large_v1.1",
                             "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
                             "microsoft/deberta-v3-large",
                             "FacebookAI/roberta-large"]

analysis.build.train_test_models(data_path, model_path, results_path, transformer_model_sources, [], [], transformer_num_epochs=6)

In [ ]:
# Running Zero-Shot LLMs
llm_model_sources = ["meta-llama/Meta-Llama-3.1-8B-Instruct", "mistralai/Ministral-8B-Instruct-2410", "Qwen/Qwen2.5-7B-Instruct"]
prompt_types = ["gpt","long","simple"]

analysis.build.train_test_models(data_path, model_path, results_path, [], prompt_types, llm_model_sources, get_llm_untrained_results=True, get_llm_trained_results=False)

In [ ]:
# Running LoRA Fine-Tuned LLMs (no class weights)

llm_model_sources = ["mistralai/Ministral-8B-Instruct-2410", "Qwen/Qwen2.5-7B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct"]
prompt_types = ["gpt","long","simple"]

analysis.build.train_test_models(data_path, model_path, results_path, [], prompt_types, llm_model_sources, get_llm_untrained_results=False, get_llm_trained_results=True, llm_num_epochs=6, llm_lr=5e-4, llm_use_class_weights=False)

In [ ]:
# predicting with the best one
best_model = "meta-llama/Meta-Llama-3.1-8B-Instruct"
best_prompt = "gpt"

#automatically does not use class weights for llm
analysis.build.train_predict(data_path, results_path, False, best_model, best_prompt, num_epochs=6, lr=5e-4, use_class_weights=False)

In [22]:
# formatting result data
analysis.data.produce_output_data("data")

Downloading...
From: https://drive.google.com/uc?id=1Jib8wKRPiFbdhB1FHEkYFraxsDY5R26c
To: c:\Users\setht\OneDrive\Desktop\nation_strategy_text_data_paper\national-strategy-text-data\data\natsec_prediction_full.csv
100%|██████████| 37.0M/37.0M [00:00<00:00, 42.0MB/s]
